## Notebook overview

This notebook introduces two important classification methods in scikit-learn:

1. **Neural Networks** using a Multi-Layer Perceptron (MLP)  
2. **Support Vector Machines (SVM)**

The dataset is stored in `vs_bank_part.csv`.

You will:
- Load the dataset from a CSV file
- Split the data using the existing `partition_Indicator` column
- Predict the binary target variable `b_tgt`
- Use two categorical predictors:
  - `cat_input1`
  - `cat_input2`
- Use twelve numeric predictors:
  - `logi_rfm1` to `logi_rfm12`
- Fit and evaluate an MLP model
- Fit and evaluate an SVM model
- Use a specified cutoff to create class predictions
- Print confusion matrices
- Report validation misclassification rate
- Plot the ROC curve
- Calculate AUC
- Report the KS / Youden statistic
- Encourage experimentation with model parameters

## Dataset instructions

The file `vs_bank_part.csv` should be downloaded from Moodle and then uploaded into the Colab session.

In Google Colab:
1. Open the notebook
2. Click the folder icon on the left panel
3. Click **Upload**
4. Upload `vs_bank_part.csv`

After uploading, the file can be loaded using:

```python
df = pd.read_csv("vs_bank_part.csv")
```

Note that uploaded files are temporary in Colab and need to be uploaded again if the session resets.

## Imports

The next cell imports the Python tools needed for this notebook.

These imports include:
- **Pandas** and **NumPy** for working with data
- preprocessing tools for categorical variables and scaling
- **MLPClassifier** for neural networks
- **SVC** for support vector machines
- evaluation tools for confusion matrix, ROC, and AUC

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# Models
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC

# Evaluation
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score


## Load the dataset

We first load the CSV file and inspect its structure.

This helps confirm:
- the file loaded correctly
- the required columns exist
- the partition column and target column are available

In [ ]:
# Load the dataset
df = pd.read_csv("vs_bank_part.csv")

# Basic inspection
print("Shape:", df.shape)
print("\nColumns:\n", df.columns.tolist())
print("\nFirst 5 rows:\n", df.head())


## Define target, predictors, and partition

The target variable is `b_tgt`.

The predictors include:
- two categorical variables: `cat_input1`, `cat_input2`
- twelve numeric variables: `logi_rfm1` to `logi_rfm12`

The dataset already contains a partition column called `partition_Indicator`, which will be used to create the training and validation datasets.

In [ ]:
# Target variable
target = "b_tgt"

# Categorical predictors
categorical_features = ["cat_input1", "cat_input2"]

# Numeric predictors
numeric_features = [
    "logi_rfm1", "logi_rfm2", "logi_rfm3", "logi_rfm4",
    "logi_rfm5", "logi_rfm6", "logi_rfm7", "logi_rfm8",
    "logi_rfm9", "logi_rfm10", "logi_rfm11", "logi_rfm12"
]

# All predictors together
all_features = categorical_features + numeric_features

print("Number of predictors:", len(all_features))
print("Predictors:", all_features)


## Create training and validation sets

Instead of using `train_test_split`, this dataset already provides a partition column.

We will:
- use rows where `partition_Indicator == "Training"` for model fitting
- use rows where `partition_Indicator == "Validation"` for evaluation

In [ ]:
# Create training and validation subsets
train_df = df[df["partition_Indicator"] == "Training"].copy()
valid_df = df[df["partition_Indicator"] == "Validation"].copy()

print("Training shape:", train_df.shape)
print("Validation shape:", valid_df.shape)

# Split into X and y
X_train = train_df[all_features]
y_train = train_df[target]

X_valid = valid_df[all_features]
y_valid = valid_df[target]


## Preprocessing

Both MLP and SVM require numeric input, and both are sensitive to variable scale.

So we will:
- one-hot encode the categorical predictors
- keep numeric predictors
- standardise the full input after encoding

This is especially important for:
- **MLP**, because neural networks train better when inputs are scaled
- **SVM**, because distance and margin calculations depend heavily on scale

In [ ]:
# One-hot encode categorical variables, keep numeric variables
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)


## Set a cutoff and helper functions

By default, binary classification often uses a cutoff of `0.50`:
- predicted probability or score >= cutoff becomes class 1
- predicted probability or score < cutoff becomes class 0

For MLP, this cutoff is applied to predicted probabilities.  
For SVM, it will be applied to decision scores.

We will also define helper functions for:
- printing the confusion matrix
- computing validation misclassification rate
- computing the KS / Youden statistic

In [ ]:
# Set the classification cutoff
cutoff = 0.50

def classify_from_cutoff(values, cutoff=0.50):
    return (values >= cutoff).astype(int)

def print_confusion_matrix(model_name, y_true, values, cutoff=0.50):
    y_pred = classify_from_cutoff(values, cutoff=cutoff)
    cm = confusion_matrix(y_true, y_pred)
    print(f"\n{model_name} confusion matrix at cutoff = {cutoff}")
    print(cm)

def misclassification_rate(y_true, values, cutoff=0.50):
    y_pred = classify_from_cutoff(values, cutoff=cutoff)
    return np.mean(y_pred != y_true)

def ks_youden_statistic(y_true, values):
    fpr, tpr, thresholds = roc_curve(y_true, values)
    ks_values = tpr - fpr
    best_idx = np.argmax(ks_values)

    return {
        "ks_youden": ks_values[best_idx],
        "best_threshold": thresholds[best_idx],
        "tpr_at_best": tpr[best_idx],
        "fpr_at_best": fpr[best_idx]
    }


## Model 1: Neural Network / MLP

We now fit a basic **Multi-Layer Perceptron (MLP)** classifier.

Important parameters:
- `hidden_layer_sizes=(10,)`: one hidden layer with 10 neurons
- `activation="relu"`: activation function used in hidden layers
- `solver="adam"`: optimisation algorithm for training
- `alpha=0.0001`: regularisation strength
- `max_iter=500`: maximum number of training iterations
- `random_state=42`: ensures reproducibility

Suggested experiments:
- change the number of neurons, for example `(5,)`, `(20,)`, `(20, 10)`
- change `activation` to `"tanh"`
- increase or decrease `alpha`
- increase `max_iter`
- compare different hidden-layer structures

In [ ]:
# Build a pipeline: preprocessing + scaling + MLP
mlp_model = Pipeline([
    ("preprocessor", preprocessor),
    ("scaler", StandardScaler()),
    ("model", MLPClassifier(
        hidden_layer_sizes=(10,),
        activation="relu",
        solver="adam",
        alpha=0.0001,
        max_iter=500,
        random_state=42
    ))
])

# Fit the model
mlp_model.fit(X_train, y_train)

# Predicted probabilities for class 1
mlp_prob = mlp_model.predict_proba(X_valid)[:, 1]

# Confusion matrix at chosen cutoff
print_confusion_matrix("MLP", y_valid, mlp_prob, cutoff=cutoff)

# Validation misclassification rate
mlp_misclass = misclassification_rate(y_valid, mlp_prob, cutoff=cutoff)
print("\nMLP validation misclassification rate:", round(mlp_misclass, 4))


## ROC, AUC, and KS / Youden for MLP

To evaluate the MLP more fully, we will:
- plot the **ROC curve**
- calculate **AUC**
- report the **KS / Youden statistic**

Interpretation:
- **ROC curve** shows the trade-off between true positive rate and false positive rate
- **AUC** summarises ROC performance into one number; higher is better
- **KS / Youden** is the maximum vertical distance between the ROC curve and the diagonal baseline, which is also equal to `TPR - FPR`

In [ ]:
# ROC curve and AUC for MLP
fpr_mlp, tpr_mlp, thresholds_mlp = roc_curve(y_valid, mlp_prob)
auc_mlp = roc_auc_score(y_valid, mlp_prob)

# KS / Youden statistic for MLP
mlp_ks_info = ks_youden_statistic(y_valid, mlp_prob)

print("MLP AUC:", round(auc_mlp, 4))
print("MLP KS / Youden statistic:", round(mlp_ks_info["ks_youden"], 4))
print("Best threshold for KS / Youden:", round(mlp_ks_info["best_threshold"], 4))

plt.figure()
plt.plot(fpr_mlp, tpr_mlp, label=f"MLP (AUC = {auc_mlp:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - MLP")
plt.legend()
plt.show()


## Model 2: Support Vector Machine

We now fit a **Support Vector Machine (SVM)** classifier.

To make training faster, we will use only a **10% sample** of the training data for the SVM model.

Important parameters:
- `kernel="linear"`: uses a linear kernel function to start with
- `C=0.1`: controls the trade-off between margin width and classification error
- `gamma="scale"`: controls how far the influence of each point reaches
- `probability=True`: allows the model to produce predicted probabilities on the 0 to 1 scale
- `random_state=42`: for reproducibility where applicable

Because `probability=True`, we can:
- apply the same probability cutoff logic as the MLP model
- interpret thresholds on the 0 to 1 scale
- report ROC, AUC, and KS / Youden using probabilities


Suggested experiments:
- change `kernel` to `"poly"` or `"rbf"` (See scikit-learn documentation for details)
- try larger or smaller `C` values such as `0.1`, `1`, `10`
- try different `gamma` values such as `"scale"`, `"auto"`, `0.01`, `0.1`
- compare linear and non-linear kernels
- change the SVM training sample fraction
- observe how parameter changes affect ROC, AUC, KS / Youden, misclassification rate, and confusion matrix

In [ ]:
# Take a 10% sample of the training data for faster SVM training
svm_train_df = train_df.sample(frac=0.10, random_state=42)

X_train_svm = svm_train_df[all_features]
y_train_svm = svm_train_df[target]

print("Original training shape:", train_df.shape)
print("SVM training sample shape:", svm_train_df.shape)

# Build a pipeline: preprocessing + scaling + SVM
svm_model = Pipeline([
    ("preprocessor", preprocessor),
    ("scaler", StandardScaler()),
    ("model", SVC(
        kernel="linear",
        C=0.1,
        gamma="scale",
        probability=True,   
        random_state=42
    ))
])

# Fit the model on the sampled training data
svm_model.fit(X_train_svm, y_train_svm)

# Predicted probabilities for class 1
svm_prob = svm_model.predict_proba(X_valid)[:, 1]

# Confusion matrix at chosen cutoff
print_confusion_matrix("SVM", y_valid, svm_prob, cutoff=cutoff)

# Validation misclassification rate
svm_misclass = misclassification_rate(y_valid, svm_prob, cutoff=cutoff)
print("\nSVM validation misclassification rate:", round(svm_misclass, 4))


## ROC, AUC, and KS / Youden for SVM

We will now evaluate SVM using the same tools:
- ROC curve
- AUC
- KS / Youden statistic

Because we are using `probability=True`, these results are based on **predicted probabilities** from the SVM.

This means:
- the reported threshold is on the **0 to 1 probability scale**
- the same cutoff logic used for MLP can also be used for SVM

This allows a more direct comparison with the MLP model.

In [ ]:
# ROC curve and AUC for SVM using decision scores
fpr_svm, tpr_svm, thresholds_svm = roc_curve(y_valid, svm_prob)
auc_svm = roc_auc_score(y_valid, svm_prob)

# KS / Youden statistic for SVM
svm_ks_info = ks_youden_statistic(y_valid, svm_prob)

print("SVM AUC:", round(auc_svm, 4))
print("SVM KS / Youden statistic:", round(svm_ks_info["ks_youden"], 4))
print("Best threshold for KS / Youden:", round(svm_ks_info["best_threshold"], 4))

plt.figure()
plt.plot(fpr_svm, tpr_svm, label=f"SVM (AUC = {auc_svm:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - SVM")
plt.legend()
plt.show()


## Compare ROC curves together

It is often useful to compare both ROC curves on one plot.

This gives a quick visual comparison of ranking performance across all possible cutoffs.

In [ ]:
plt.figure()
plt.plot(fpr_mlp, tpr_mlp, label=f"MLP (AUC = {auc_mlp:.4f})")
plt.plot(fpr_svm, tpr_svm, label=f"SVM (AUC = {auc_svm:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Comparison: MLP vs SVM")
plt.legend()
plt.show()


## Compare AUC, misclassification rate, and KS / Youden

We now compare the two models using:
- **AUC**
- **validation misclassification rate** at the chosen cutoff
- **KS / Youden statistic**

Higher AUC and KS / Youden are better.  
Lower misclassification rate is better.

In [ ]:
summary_results = pd.DataFrame({
    "Model": ["MLP", "SVM"],
    "AUC": [auc_mlp, auc_svm],
    "Validation Misclassification Rate": [mlp_misclass, svm_misclass],
    "KS / Youden Statistic": [
        mlp_ks_info["ks_youden"],
        svm_ks_info["ks_youden"]
    ],
    "Best KS / Youden Threshold": [
        mlp_ks_info["best_threshold"],
        svm_ks_info["best_threshold"]
    ]
})

print(summary_results.sort_values("AUC", ascending=False))


## Suggested experiments

Suggested student experiments:

### For MLP
- change `hidden_layer_sizes`
- change `activation`
- change `alpha`
- increase `max_iter`

### For SVM
- change `kernel`
- change `C`
- change `gamma`
- change the training sample fraction for faster or slower SVM fitting

### For both
- change the classification cutoff
- compare misclassification rate at different cutoffs
- compare confusion matrices at different cutoffs
- compare ROC, AUC, and KS / Youden
- discuss which model seems better for this dataset and why